# Análisis de Estabilidad de Talud — FEM con Criterio de Mohr-Coulomb
## Elemento CST (Triángulo de Deformación Constante) · Malla Triangular · Esfuerzo Plano

---

**Universidad Nacional de Colombia sede Manizales**

---

## El problema

Se tiene un **talud homogéneo** de altura $H = 10$ m con inclinación $2H:1V$. El suelo presenta las siguientes propiedades:

| Parámetro | Símbolo | Valor | Unidad |
|-----------|---------|-------|--------|
| Módulo de Young | $E$ | 25 000 | kPa |
| Relación de Poisson | $\nu$ | 0.30 | — |
| Peso unitario | $\gamma$ | 18 | kN/m³ |
| Cohesión efectiva | $c'$ | 20 | kPa |
| Ángulo de fricción | $\phi'$ | 25 | ° |

### Formulación FEM — Elemento CST (3 nodos, esfuerzo plano)

Para cada elemento triangular con nodos $(x_1,y_1)$, $(x_2,y_2)$, $(x_3,y_3)$:

**Área del elemento:**
$$A_e = \frac{1}{2}\left|(x_2-x_1)(y_3-y_1) - (x_3-x_1)(y_2-y_1)\right|$$

**Matriz de deformación-desplazamiento** (constante dentro del elemento):
$$\mathbf{B} = \frac{1}{2A_e}\begin{bmatrix} b_1 & 0 & b_2 & 0 & b_3 & 0 \\ 0 & c_1 & 0 & c_2 & 0 & c_3 \\ c_1 & b_1 & c_2 & b_2 & c_3 & b_3 \end{bmatrix}$$

con $b_i = y_j - y_k$, $c_i = x_k - x_j$ (permutación cíclica de $i,j,k$).

**Matriz constitutiva** (esfuerzo plano):
$$\mathbf{D} = \frac{E}{(1+\nu)(1-2\nu)}\begin{bmatrix}1-\nu & \nu & 0 \\ \nu & 1-\nu & 0 \\ 0 & 0 & \frac{1-2\nu}{2}\end{bmatrix}$$

**Rigidez del elemento** (por unidad de espesor, $t=1$ m):
$$\mathbf{k}_e = t\,A_e\,\mathbf{B}^T\mathbf{D}\mathbf{B}$$

**Vector de fuerzas de cuerpo** (peso propio, distribución uniforme a los 3 nodos):
$$\mathbf{f}_e^{\text{grav}} = \frac{\gamma\,A_e\,t}{3}\begin{bmatrix}0 \\ -1 \\ 0 \\ -1 \\ 0 \\ -1\end{bmatrix}$$

**Esfuerzos en el elemento:**
$$\boldsymbol{\sigma}_e = \mathbf{D}\mathbf{B}\,\mathbf{u}_e = \{\sigma_x,\, \sigma_y,\, \tau_{xy}\}^T$$

### Criterio de Mohr-Coulomb — Factor de Seguridad local

En cada punto del dominio, se computan los esfuerzos principales:
$$\sigma_{1,2} = \frac{\sigma_x+\sigma_y}{2} \pm \sqrt{\left(\frac{\sigma_x-\sigma_y}{2}\right)^2+\tau_{xy}^2}$$

La resistencia al corte disponible (radio del círculo de Mohr en la envolvente) es:
$$\tau_{\text{disp}} = c'\cos\phi' + \left(-\frac{\sigma_x+\sigma_y}{2}\right)\sin\phi'$$

donde el signo negativo convierte de la convención FEM (tracción $+$) a la convención geotécnica (compresión $+$).

El **Factor de Seguridad local** (radio del círculo respecto a la envolvente):
$$FS_{\text{local}} = \frac{\tau_{\text{disp}}}{\tau_{\text{máx}}} = \frac{c'\cos\phi' - \sigma_m\sin\phi'}{\sqrt{\left(\frac{\sigma_x-\sigma_y}{2}\right)^2+\tau_{xy}^2}}$$

$FS_{\text{local}} < 1$ indica zona plastificada (la demanda supera la resistencia).

## Importamos librerías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib.path import Path
from matplotlib.patches import FancyArrowPatch
from matplotlib.colors import TwoSlopeNorm
from scipy.spatial import Delaunay
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.25
})

print('Librerías importadas.')

## Parámetros del problema

In [ ]:
# ── Propiedades elásticas del suelo ───────────────────────────────────────
E_mod   = 25_000.0   # Módulo de Young [kPa]
nu      = 0.30       # Relación de Poisson [-]

# ── Criterio de Mohr-Coulomb ──────────────────────────────────────────────
gamma   = 18.0       # Peso unitario [kN/m³]
c_prim  = 20.0       # Cohesión efectiva [kPa]
phi_deg = 25.0       # Ángulo de fricción efectivo [°]
phi     = np.radians(phi_deg)

# ── Geometría del talud ───────────────────────────────────────────────────
H       = 10.0       # Altura [m]
x_max   = 60.0       # Ancho del dominio [m]
x_toe   = 20.0       # Inicio del talud (cresta) [m]
x_base  = 40.0       # Pie del talud [m]

# Vértices del polígono del dominio
# (0,0) → (0,10) → (20,10) → (40,0) → (60,0) → cierre
poly_pts = np.array([[0., 0.], [0., 10.], [20., 10.], [40., 0.], [60., 0.]])
slope_path = Path(np.vstack([poly_pts, poly_pts[0]]))

def in_domain(pts, r=0.0):
    """True si el punto está dentro del polígono del dominio."""
    return slope_path.contains_points(np.atleast_2d(pts), radius=r)

def surf_y(x):
    """Elevación superficial del terreno."""
    x = np.asarray(x, float)
    return np.where(x <= x_toe, H,
           np.where(x <= x_base, H*(x_base-x)/(x_base-x_toe), 0.))

print('=' * 55)
print('  PARÁMETROS DEL PROBLEMA')
print('=' * 55)
print(f"  E  = {E_mod:,.0f} kPa     ν  = {nu}")
print(f"  γ  = {gamma} kN/m³    c' = {c_prim} kPa    φ' = {phi_deg}°")
print(f"  Dominio: [{0},{x_max}] m × [0,{H}] m (forma de talud)")
print(f"  Método:  FEM — Elemento CST — Esfuerzo plano")
print('=' * 55)

## Generación de la malla triangular

Se construye una malla de **elementos triangulares CST** mediante triangulación de Delaunay. Los nodos se colocan:

- Explícitamente sobre los **bordes del dominio** (izquierdo, plataforma alta, cara del talud, base)
- En una **grilla interior** filtrada por el polígono del dominio

Esto garantiza que la cara del talud quede bien representada y que los bordes tengan nodos para aplicar condiciones de frontera.

In [ ]:
def generate_mesh(n_bound=25, nx=30, ny=12):
    """
    Malla triangular Delaunay con nodos de borde explícitos.

    Parámetros
    ----------
    n_bound : nodos por arista de borde
    nx, ny  : grilla interior de puntos candidatos
    """
    nb = n_bound

    # ── Bordes del dominio ────────────────────────────────────────────────
    left   = np.c_[np.zeros(nb),         np.linspace(0, 10, nb)]   # lado izquierdo
    top_pl = np.c_[np.linspace(0, 20,nb),np.ones(nb)*10]            # plataforma alta
    slope  = np.c_[np.linspace(20,40,nb),np.linspace(10, 0, nb)]    # cara del talud
    bot_r  = np.c_[np.linspace(40,60,nb),np.zeros(nb)]              # plataforma baja
    bot_l  = np.c_[np.linspace(0, 60,nb*2), np.zeros(nb*2)]         # base completa

    # ── Puntos interiores ─────────────────────────────────────────────────
    xs = np.linspace(1, 59, nx)
    ys = np.linspace(0.4, 9.5, ny)
    XX, YY = np.meshgrid(xs, ys)
    inter = np.c_[XX.ravel(), YY.ravel()]
    inter = inter[in_domain(inter, -0.5)]   # sólo estrictamente adentro

    # ── Triangulación de Delaunay ─────────────────────────────────────────
    pts = np.vstack([left, top_pl, slope, bot_r, bot_l, inter])
    pts = np.unique(pts.round(8), axis=0)

    tri = Delaunay(pts)
    els = tri.simplices.copy()

    # Filtrar triángulos con centroide fuera del dominio
    cents = pts[els].mean(axis=1)
    els   = els[in_domain(cents, -0.05)]

    # Renumerar nodos para usar sólo los referenciados
    used  = np.unique(els)
    remap = np.full(len(pts), -1)
    remap[used] = np.arange(len(used))
    return pts[used], remap[els]


nodes, elems = generate_mesh(n_bound=25, nx=30, ny=12)
n_nod, n_el = len(nodes), len(elems)

print('=' * 55)
print('  MALLA TRIANGULAR')
print('=' * 55)
print(f'  Nodos:     {n_nod}')
print(f'  Elementos: {n_el}')
print(f'  DOF:       {2*n_nod} (2 por nodo)')

# Identificar nodos de borde para condiciones de frontera
tol_bc = 0.01
mask_bot  = nodes[:,1] < tol_bc       # base → empotramiento
mask_left = nodes[:,0] < tol_bc       # lado izquierdo → rodillo en x
n_bot  = mask_bot.sum()
n_left = mask_left.sum()
print(f'  Nodos en la base:           {n_bot}')
print(f'  Nodos en el lado izquierdo: {n_left}')
print('=' * 55)

## Ensamblaje del sistema FEM

### Matriz constitutiva $\mathbf{D}$ (esfuerzo plano)

En esfuerzo plano ($\sigma_z = \tau_{xz} = \tau_{yz} = 0$), la relación esfuerzo-deformación es:

$$\boldsymbol{\sigma} = \mathbf{D}\,\boldsymbol{\varepsilon}, \quad
\mathbf{D} = \frac{E}{(1+\nu)(1-2\nu)}\begin{bmatrix}1-\nu & \nu & 0 \\ \nu & 1-\nu & 0 \\ 0 & 0 & \frac{1-2\nu}{2}\end{bmatrix}$$

In [ ]:
# ── Matriz constitutiva D — Esfuerzo plano ─────────────────────────────────
lam = E_mod * nu / ((1 + nu) * (1 - 2*nu))   # primer parámetro de Lamé
mu  = E_mod / (2 * (1 + nu))                  # módulo de corte G

D = np.array([
    [lam + 2*mu, lam,       0  ],
    [lam,        lam + 2*mu, 0  ],
    [0,          0,         mu  ]
])

print(f"Parámetros de Lamé:  λ = {lam:.1f} kPa   G (μ) = {mu:.1f} kPa")
print(f"\nMatriz D [kPa]:\n{D.round(0)}")


# ── Función: rigidez + carga de un elemento CST ────────────────────────────
def cst_ke_fe(xe):
    """
    Rigidez ke (6×6) y vector de carga gravitacional fe (6,) del elemento CST.

    Parámetros
    ----------
    xe : array (3,2) con coordenadas de los 3 nodos

    Retorna
    -------
    ke : matriz de rigidez del elemento (6×6)
    fe : vector de fuerzas de cuerpo (6,)  — peso propio
    B  : matriz cinemática (3×6)
    A  : área del elemento [m²]
    """
    x1, y1 = xe[0];  x2, y2 = xe[1];  x3, y3 = xe[2]

    # Área con signo → el valor absoluto es el área real
    A = 0.5 * abs((x2 - x1)*(y3 - y1) - (x3 - x1)*(y2 - y1))
    if A < 1e-14:
        return np.zeros((6,6)), np.zeros(6), np.zeros((3,6)), A

    # Coeficientes b_i y c_i (derivadas de las funciones de forma)
    b1 = y2 - y3;  b2 = y3 - y1;  b3 = y1 - y2
    c1 = x3 - x2;  c2 = x1 - x3;  c3 = x2 - x1

    # Matriz B (3×6) — deformaciones = B · u_e
    B = np.array([
        [b1,  0,  b2,  0,  b3,  0 ],
        [ 0, c1,   0, c2,   0, c3 ],
        [c1, b1,  c2, b2,  c3, b3 ]
    ]) / (2 * A)

    # Rigidez del elemento ke = t·A·B^T·D·B  (t = 1 m, esfuerzo plano)
    ke = A * (B.T @ D @ B)

    # Fuerzas de cuerpo — peso propio distribuido uniformemente a los 3 nodos
    fe = np.zeros(6)
    fe[1] = fe[3] = fe[5] = -gamma * A / 3.0   # componente Y = -γ·V/3

    return ke, fe, B, A


# ── Ensamblaje global ──────────────────────────────────────────────────────
n_dof = 2 * n_nod
K_glob = np.zeros((n_dof, n_dof))
F_glob = np.zeros(n_dof)
B_list = []   # matrices B por elemento (para recuperar esfuerzos)
A_list = []   # áreas por elemento

for conn in elems:
    ke, fe, B, A = cst_ke_fe(nodes[conn])

    # Mapeo DOF: nodo n → DOF [2n, 2n+1]
    dofs = np.array([
        2*conn[0], 2*conn[0]+1,
        2*conn[1], 2*conn[1]+1,
        2*conn[2], 2*conn[2]+1
    ])

    K_glob[np.ix_(dofs, dofs)] += ke
    F_glob[dofs]               += fe
    B_list.append(B)
    A_list.append(A)

print(f"\nMatriz global K ensamblada: {K_glob.shape}")
print(f"Ancho de banda estimado: {n_dof} DOF")

## Condiciones de frontera y resolución

| Borde | Condición | Descripción |
|-------|-----------|-------------|
| Base $y = 0$ | $u_x = u_y = 0$ | Empotramiento (roca firme) |
| Lado izquierdo $x = 0$ | $u_x = 0$ | Rodillo — libre en $y$ |
| Superficie libre | Sin restricción | Carga superficial = 0 |

Se resuelve el sistema reducido $\mathbf{K}_{\text{libre}}\,\mathbf{u}_{\text{libre}} = \mathbf{F}_{\text{libre}}$ directamente.

In [ ]:
# ── Grados de libertad fijos ───────────────────────────────────────────────
fixed_dofs = set()
tol_bc = 0.01

for i, (x, y) in enumerate(nodes):
    if y < tol_bc:                     # base: empotramiento
        fixed_dofs.add(2*i)             #  → ux = 0
        fixed_dofs.add(2*i + 1)         #  → uy = 0
    if x < tol_bc:                     # lado izquierdo: rodillo
        fixed_dofs.add(2*i)             #  → ux = 0

fixed_dofs = sorted(fixed_dofs)
free_dofs  = [d for d in range(n_dof) if d not in fixed_dofs]

print(f"DOF fijos:  {len(fixed_dofs)}")
print(f"DOF libres: {len(free_dofs)}")

# ── Resolución del sistema lineal ─────────────────────────────────────────
K_ff = K_glob[np.ix_(free_dofs, free_dofs)]
F_f  = F_glob[free_dofs]
u_f  = np.linalg.solve(K_ff, F_f)

# Vector completo de desplazamientos
u_all = np.zeros(n_dof)
u_all[free_dofs] = u_f

ux = u_all[0::2]   # desplazamientos horizontales [m]
uy = u_all[1::2]   # desplazamientos verticales [m]
U_mag = np.sqrt(ux**2 + uy**2)   # magnitud [m]

print(f"\nDesplazamiento vertical máx. (asentamiento):  {uy.min()*1000:.1f} mm")
print(f"Desplazamiento horizontal máx.:               {np.abs(ux).max()*1000:.1f} mm")
print(f"Magnitud máx. del desplazamiento:             {U_mag.max()*1000:.1f} mm")

## Recuperación de esfuerzos y criterio de Mohr-Coulomb

In [ ]:
# ── Esfuerzos por elemento ─────────────────────────────────────────────────
sig_el = np.zeros((n_el, 3))  # [σx, σy, τxy] por elemento

for e, conn in enumerate(elems):
    dofs = np.array([
        2*conn[0], 2*conn[0]+1,
        2*conn[1], 2*conn[1]+1,
        2*conn[2], 2*conn[2]+1
    ])
    sig_el[e] = D @ B_list[e] @ u_all[dofs]   # σ = D · B · u_e


# ── Promedio a nodos (suavizado para contornos) ────────────────────────────
#   Cada nodo recibe el promedio de los esfuerzos de sus elementos vecinos.
sig_nd  = np.zeros((n_nod, 3))
cnt_nd  = np.zeros(n_nod)

for e, conn in enumerate(elems):
    for n in conn:
        sig_nd[n] += sig_el[e]
        cnt_nd[n] += 1

sig_nd /= cnt_nd[:, None]

sx  = sig_nd[:, 0]    # esfuerzo normal horizontal [kPa]  (tracción +)
sy  = sig_nd[:, 1]    # esfuerzo normal vertical   [kPa]  (tracción +)
txy = sig_nd[:, 2]    # esfuerzo cortante          [kPa]


# ── Esfuerzos principales ──────────────────────────────────────────────────
sm   = (sx + sy) / 2                                # centro del círculo de Mohr
R_mc = np.sqrt(((sx - sy)/2)**2 + txy**2)           # radio = τ_máx
sig1 = sm + R_mc                                    # esfuerzo principal mayor
sig2 = sm - R_mc                                    # esfuerzo principal menor

# Ángulo de los esfuerzos principales [°]
theta_p = 0.5 * np.degrees(np.arctan2(2*txy, sx - sy + 1e-12))


# ── Factor de Seguridad local (Mohr-Coulomb) ──────────────────────────────
#   Convención: FEM usa tracción+; suelos usan compresión+
#   → σ_m_suelo = -σ_m_FEM
#
#   τ_disp = c'·cos(φ') + σ_m_suelo·sin(φ') = c'·cos(φ') - σ_m·sin(φ')
#   FS_local = τ_disp / τ_máx

tau_disp = c_prim * np.cos(phi) - sm * np.sin(phi)  # resistencia disponible
FS_nd    = np.where(R_mc > 1e-8, tau_disp / R_mc, 10.0)
FS_nd    = np.clip(FS_nd, 0.0, 10.0)

print('Resumen de esfuerzos (nodos):')
print(f'  σx : [{sx.min():.1f}, {sx.max():.1f}] kPa')
print(f'  σy : [{sy.min():.1f}, {sy.max():.1f}] kPa  (≈ -γH = {-gamma*H:.0f} kPa en la base)')
print(f'  τxy: [{txy.min():.1f}, {txy.max():.1f}] kPa')
print(f'  σ1 : [{sig1.min():.1f}, {sig1.max():.1f}] kPa')
print(f'  σ2 : [{sig2.min():.1f}, {sig2.max():.1f}] kPa')
print()
print(f'  FS local mínimo: {FS_nd.min():.2f}')
print(f'  FS local máximo: {FS_nd[FS_nd < 9.9].max():.2f}')
print(f'  Nodos con FS < 1.5: {(FS_nd < 1.5).sum()} / {n_nod}')
print(f'  Nodos con FS < 1.0: {(FS_nd < 1.0).sum()} / {n_nod}')

## Gráficas

### Gráfica 1 — Malla triangular CST

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

# Malla
triang = mtri.Triangulation(nodes[:,0], nodes[:,1], elems)
ax.triplot(triang, color='steelblue', lw=0.4, alpha=0.6)

# Relleno del dominio
x_surf = np.linspace(0, x_max, 400)
ax.fill_between(x_surf, 0, surf_y(x_surf), color='#c8a96e', alpha=0.25, zorder=0)

# Perfil del terreno
ax.plot(x_surf, surf_y(x_surf), 'k-', lw=2, label='Perfil del terreno')
ax.axhline(0, color='saddlebrown', lw=1, linestyle='--', alpha=0.5)

# Nodos de borde coloreados
mask_bot  = nodes[:,1] < tol_bc
mask_left = nodes[:,0] < tol_bc
mask_surf = ~mask_bot & ~mask_left & in_domain(nodes, r=0.05) & ~in_domain(nodes, r=-0.5)

ax.scatter(nodes[mask_bot, 0],  nodes[mask_bot, 1],  s=18, color='tomato',
           zorder=5, label=f'Base empotrada ({mask_bot.sum()} nodos)')
ax.scatter(nodes[mask_left, 0], nodes[mask_left, 1], s=18, color='royalblue',
           zorder=5, label=f'Borde izquierdo — rodillo ({mask_left.sum()} nodos)')
ax.scatter(nodes[~mask_bot & ~mask_left, 0],
           nodes[~mask_bot & ~mask_left, 1],
           s=5, color='navy', alpha=0.5, label='Nodos interiores')

ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
ax.set_xlim(-2, x_max+2); ax.set_ylim(-1, 13)
ax.set_aspect('equal')
ax.set_title(f'Gráfica 1 — Malla triangular CST   ({n_nod} nodos · {n_el} elementos)')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()

### Gráfica 2 — Campo de desplazamientos (malla deformada)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Malla deformada ───────────────────────────────────────────────────────
scale = 30   # factor de amplificación visual
x_def = nodes[:,0] + scale * ux
y_def = nodes[:,1] + scale * uy

tri_def = mtri.Triangulation(x_def, y_def, elems)
tri_ori = mtri.Triangulation(nodes[:,0], nodes[:,1], elems)

ax1 = axes[0]
ax1.triplot(tri_ori, color='lightgray', lw=0.5, alpha=0.6, label='Original')
ax1.triplot(tri_def, color='steelblue', lw=0.6, label=f'Deformada ×{scale}')
ax1.set_xlabel('x [m]'); ax1.set_ylabel('y [m]')
ax1.set_aspect('equal')
ax1.set_title(f'Malla deformada (amplificada ×{scale})')
ax1.legend(fontsize=9)

# ── Magnitud de desplazamiento ────────────────────────────────────────────
ax2 = axes[1]
im = ax2.tricontourf(tri_ori, U_mag * 1000, levels=20, cmap='YlOrRd')
ax2.tricontour(tri_ori,  U_mag * 1000, levels=8, colors='k', linewidths=0.5, alpha=0.5)
cbar = plt.colorbar(im, ax=ax2)
cbar.set_label('|U| [mm]')

# Vectores de desplazamiento (submuestreo)
step = max(1, n_nod // 60)
idx  = np.arange(0, n_nod, step)
ax2.quiver(nodes[idx,0], nodes[idx,1],
           ux[idx]*scale*2, uy[idx]*scale*2,
           color='navy', scale=1.5, width=0.003, alpha=0.6)

ax2.plot(x_surf, surf_y(x_surf), 'k-', lw=1.5)
ax2.set_xlabel('x [m]'); ax2.set_ylabel('y [m]')
ax2.set_aspect('equal')
ax2.set_title(f'Magnitud de desplazamiento |U| [mm]  (max = {U_mag.max()*1000:.1f} mm)')

fig.suptitle('Gráfica 2 — Campo de desplazamientos por peso propio', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### Gráfica 3 — Esfuerzo vertical $\sigma_y$ y esfuerzo horizontal $\sigma_x$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

triang = mtri.Triangulation(nodes[:,0], nodes[:,1], elems)

def stress_plot(ax, triang, vals, title, cmap, label):
    """Función auxiliar para graficar contornos de esfuerzo."""
    vmax = np.percentile(np.abs(vals), 98)
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    im = ax.tricontourf(triang, vals, levels=25, cmap=cmap, norm=norm)
    ax.tricontour(triang, vals, levels=10, colors='k', linewidths=0.4, alpha=0.4)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(label)
    ax.plot(x_surf, surf_y(x_surf), 'k-', lw=1.5)
    ax.axhline(0, color='k', lw=0.5, alpha=0.3)
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    ax.set_aspect('equal')
    ax.set_title(title)

stress_plot(axes[0], triang, sy, 'Esfuerzo vertical  sigma_y  [kPa]',
            'RdBu_r', 'sigma_y [kPa]  (− compresión)')
stress_plot(axes[1], triang, sx, 'Esfuerzo horizontal  sigma_x  [kPa]',
            'RdBu_r', 'sigma_x [kPa]  (− compresión)')

fig.suptitle('Gráfica 3 — Esfuerzos normales (convención FEM: tracción = +)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Verificación litostática
nod_base_center = nodes[(nodes[:,0] > 5) & (nodes[:,0] < 15) & (nodes[:,1] < 0.5)]
idx_bc = np.where((nodes[:,0] > 5) & (nodes[:,0] < 15) & (nodes[:,1] < 0.5))[0]
if len(idx_bc) > 0:
    sy_base = sy[idx_bc].mean()
    print(f'Verificacion litostática:')
    print(f'  σy en la base (x~10m):    {sy_base:.1f} kPa')
    print(f'  Valor teórico (-γH):       {-gamma*H:.1f} kPa')
    print(f'  Error relativo:            {abs(sy_base+gamma*H)/(gamma*H)*100:.1f}%')

### Gráfica 4 — Esfuerzo cortante $\tau_{xy}$ y esfuerzo de corte máximo $\tau_{\max}$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── τxy ───────────────────────────────────────────────────────────────────
ax1 = axes[0]
lim = np.percentile(np.abs(txy), 98)
norm_t = TwoSlopeNorm(vmin=-lim, vcenter=0, vmax=lim)
im1 = ax1.tricontourf(triang, txy, levels=25, cmap='PiYG', norm=norm_t)
ax1.tricontour(triang, txy, levels=[0], colors='k', linewidths=1.5)
cbar1 = plt.colorbar(im1, ax=ax1)
cbar1.set_label('tau_xy [kPa]')
ax1.plot(x_surf, surf_y(x_surf), 'k-', lw=1.5)
ax1.set_xlabel('x [m]'); ax1.set_ylabel('y [m]')
ax1.set_aspect('equal')
ax1.set_title('Esfuerzo cortante tau_xy [kPa]')

# ── τ_max ─────────────────────────────────────────────────────────────────
ax2 = axes[1]
im2 = ax2.tricontourf(triang, R_mc, levels=25, cmap='hot_r')
ax2.tricontour(triang, R_mc, levels=8, colors='k', linewidths=0.4, alpha=0.5)
cbar2 = plt.colorbar(im2, ax=ax2)
cbar2.set_label('tau_max [kPa]')
ax2.plot(x_surf, surf_y(x_surf), 'k-', lw=1.5)
ax2.set_xlabel('x [m]'); ax2.set_ylabel('y [m]')
ax2.set_aspect('equal')
ax2.set_title('Corte maximo  tau_max = (sigma_1 - sigma_2)/2  [kPa]')

fig.suptitle('Gráfica 4 — Esfuerzos cortantes', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### Gráfica 5 — Factor de Seguridad local (Criterio Mohr-Coulomb) y zonas en riesgo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── FS local ──────────────────────────────────────────────────────────────
ax1 = axes[0]
FS_plot = np.clip(FS_nd, 0.5, 4.0)
norm_fs = TwoSlopeNorm(vmin=0.5, vcenter=1.5, vmax=4.0)
im1 = ax1.tricontourf(triang, FS_plot, levels=25, cmap='RdYlGn', norm=norm_fs)
cs  = ax1.tricontour(triang,  FS_plot, levels=[1.0, 1.2, 1.5, 2.0, 3.0],
                     colors='k', linewidths=0.9)
ax1.clabel(cs, fmt='FS=%.1f', fontsize=8)
cbar1 = plt.colorbar(im1, ax=ax1)
cbar1.set_label('FS local')

# Resaltar zona critica (FS < 1.5)
zona_crit = FS_nd < 1.5
if zona_crit.sum() > 0:
    ax1.scatter(nodes[zona_crit,0], nodes[zona_crit,1],
                s=25, color='red', zorder=5, alpha=0.7,
                label=f'FS < 1.5  ({zona_crit.sum()} nodos)')

ax1.plot(x_surf, surf_y(x_surf), 'k-', lw=1.5)
ax1.set_xlabel('x [m]'); ax1.set_ylabel('y [m]')
ax1.set_aspect('equal')
ax1.set_title(f'Factor de Seguridad local  FS_min = {FS_nd.min():.2f}')
ax1.legend(fontsize=9)

# ── Envolvente de Mohr-Coulomb con todos los nodos ────────────────────────
ax2 = axes[1]

# sigma normal en la plano de falla: se usa -sm (compresion +)
sigma_n_soil = -sm   # compresión positiva
tau_avail    = c_prim + sigma_n_soil * np.tan(phi)

sv = np.linspace(sigma_n_soil.min()*0.9, sigma_n_soil.max()*1.1, 300)
tv = c_prim + sv * np.tan(phi)
ax2.plot(sv, tv, 'k-', lw=2.5,
         label=f"Envolvente MC  (c'={c_prim} kPa, phi'={phi_deg}°)")
ax2.fill_between(sv[sv>=0], tv[sv>=0], alpha=0.08, color='green')

sc = ax2.scatter(sigma_n_soil, R_mc, c=FS_nd, cmap='RdYlGn',
                 vmin=0.5, vmax=3.0, s=15, alpha=0.7,
                 edgecolors='none', label='Nodos FEM')
cbar2 = plt.colorbar(sc, ax=ax2)
cbar2.set_label('FS local')

ax2.axhline(0, color='k', lw=0.5)
ax2.axvline(0, color='k', lw=0.5)
ax2.set_xlabel("sigma_n [kPa]  (compresion +)")
ax2.set_ylabel("tau_max [kPa]")
ax2.set_title("Envolvente de Mohr-Coulomb — todos los nodos")
ax2.legend(fontsize=9)

fig.suptitle('Gráfica 5 — Criterio de Mohr-Coulomb y Factor de Seguridad local',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### Gráfica 6 — Trayectorias de esfuerzos principales $\sigma_1$ y $\sigma_2$

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

# Fondo: esfuerzo cortante máximo
im = ax.tricontourf(triang, R_mc, levels=20, cmap='Blues', alpha=0.55)
plt.colorbar(im, ax=ax, label='tau_max [kPa]')

# Trayectorias de esfuerzos principales (crucetas en cada nodo)
# Se grafica como líneas proporcionales a la magnitud
step = max(1, n_nod // 120)
idx_q = np.arange(0, n_nod, step)

scale_s = 0.6   # escala de las crucetas [m/kPa]

for i in idx_q:
    x0, y0 = nodes[i]
    th = np.radians(theta_p[i])
    
    # Dirección del esfuerzo principal 1 (normalizada por magnitud)
    mag1 = min(abs(sig1[i]) * scale_s * 0.04, 1.5)
    mag2 = min(abs(sig2[i]) * scale_s * 0.04, 1.5)
    
    dx1, dy1 = mag1 * np.cos(th),   mag1 * np.sin(th)
    dx2, dy2 = mag2 * np.cos(th+np.pi/2), mag2 * np.sin(th+np.pi/2)
    
    # sigma1 (color azul si compresión, rojo si tracción)
    c1 = 'tomato' if sig1[i] > 0 else 'steelblue'
    ax.plot([x0-dx1, x0+dx1], [y0-dy1, y0+dy1], '-', color=c1, lw=1.2, alpha=0.8)
    
    # sigma2 (cruzado)
    c2 = 'tomato' if sig2[i] > 0 else 'steelblue'
    ax.plot([x0-dx2, x0+dx2], [y0-dy2, y0+dy2], '-', color=c2, lw=0.8, alpha=0.6)

# Leyenda manual
from matplotlib.lines import Line2D
leg = [
    Line2D([0],[0], color='tomato',   lw=2, label='Traccion (sigma > 0)'),
    Line2D([0],[0], color='steelblue',lw=2, label='Compresion (sigma < 0)')
]

ax.plot(x_surf, surf_y(x_surf), 'k-', lw=2)
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
ax.set_xlim(-1, x_max+1)
ax.set_ylim(-0.5, 12)
ax.set_aspect('equal')
ax.set_title('Gráfica 6 — Trayectorias de esfuerzos principales  (largo proporcional a magnitud)')
ax.legend(handles=leg, fontsize=9, loc='upper right')
plt.tight_layout()
plt.show()

## Resumen de resultados

In [ ]:
# Nodo con FS mínimo
i_min = np.argmin(FS_nd)
x_min, y_min = nodes[i_min]

print('=' * 60)
print('  RESUMEN COMPLETO — FEM TALUD MOHR-COULOMB')
print('=' * 60)
print()
print('  MODELO FEM')
print(f'    Elemento: CST (Triángulo deformacion constante)')
print(f'    Formulacion: Esfuerzo plano')
print(f'    Nodos: {n_nod}   |   Elementos: {n_el}   |   DOF: {2*n_nod}')
print()
print('  PROPIEDADES DEL MATERIAL')
print(f'    E = {E_mod:,.0f} kPa   nu = {nu}   gamma = {gamma} kN/m3')
print(f"    c' = {c_prim} kPa   phi' = {phi_deg}°")
print()
print('  DESPLAZAMIENTOS (peso propio)')
print(f'    Asentamiento max.: {uy.min()*1000:.1f} mm')
print(f'    Desplaz. horiz. max.: {np.abs(ux).max()*1000:.1f} mm')
print()
print('  ESFUERZOS')
_ib = np.where((nodes[:,0]>5)&(nodes[:,0]<15)&(nodes[:,1]<0.5))[0]
print(f'    sigma_y base (x~10m, y~0): {sy[_ib].mean():+.1f} kPa  (teorico: {-gamma*H:.0f} kPa)')
print(f'    tau_xy maximo: {np.abs(txy).max():.1f} kPa  (cerca de la cara del talud)')
print(f'    sigma_1 maximo (traccion): {sig1.max():.1f} kPa')
print(f'    sigma_2 minimo (compresion): {sig2.min():.1f} kPa')
print()
print('  CRITERIO MOHR-COULOMB')
print(f'    FS local minimo: {FS_nd.min():.3f}  en ({x_min:.1f}, {y_min:.1f}) m')
print(f'    Nodos con FS < 1.5: {(FS_nd<1.5).sum()} ({100*(FS_nd<1.5).mean():.1f}%)')
print(f'    Nodos con FS < 1.0: {(FS_nd<1.0).sum()} ({100*(FS_nd<1.0).mean():.1f}%)')
print()
if FS_nd.min() >= 1.5:
    print('  INTERPRETACION: Talud ESTABLE — ningun nodo en falla, FS_min > 1.5')
elif FS_nd.min() >= 1.0:
    print('  INTERPRETACION: Talud MARGINALMENTE ESTABLE — zona critica presente')
else:
    print('  INTERPRETACION: ZONA DE FALLA POTENCIAL — FS local < 1.0 detectado')
print('=' * 60)